# HERMES AI Model Lab — QLoRA + held-out evaluation

Base: Qwen/Qwen2.5-1.5B-Instruct. Eight command labels follow the existing Greenhouse lab: FIND, INTAKE, RETURN, STATUS, VERIFY, REPORT, DISCOVER, EXECUTE. This model classifies requests; it does not execute tools or grant authority.

Synthetic data: 192 training rows (96 phrases plus prefix variants), 16 validation, 40 independent test phrases. A score here is a Lab result, not production or phone acceptance.

This run uses local staging under `/content/HERMES_AI_Model_Lab`. Checkpoints, adapter, tokenizer and evidence are archived, downloaded, uploaded through the existing Drive connector, and independently read back with hashes. Runtime-local files are temporary: complete the export before disconnecting. Drive mounting is not required for this path. Existing Colab credits are used; no subscription, purchase, GitHub merge or deployment occurs.


In [ ]:
# 1. GPU and dependencies; install only missing packages
import sys, subprocess, importlib.util
for package in ['transformers', 'peft', 'accelerate', 'datasets', 'bitsandbytes', 'safetensors']:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
import torch
assert torch.cuda.is_available(), 'No GPU: reconnect an available authorized GPU. Do not buy or upgrade automatically.'
print('Python:', sys.version.split()[0], 'Torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0), 'VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory/2**30,2))


In [ ]:
# 2. Stage only Lab artifacts locally; upload via the existing Drive connector
from pathlib import Path
import json, hashlib, time, os
from datetime import datetime, timezone
from importlib.metadata import version
run_id=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
run_dir=Path('/content/HERMES_AI_Model_Lab')/run_id
run_dir.mkdir(parents=True,exist_ok=False)
probe=run_dir/'write-readback.txt'
probe.write_text('HERMES staging verified',encoding='utf-8')
assert probe.read_text()=='HERMES staging verified'
MODEL_ID='Qwen/Qwen2.5-1.5B-Instruct'
SEED=42
SYSTEM='Classify the user request into one command. Reply with command name only. Commands: FIND (search work or records), INTAKE (receive a new request or brief), RETURN (send results back), STATUS (read progress), VERIFY (check correctness or evidence), REPORT (write a summary or report), DISCOVER (find suitable tools or capabilities), EXECUTE (perform an assigned authorized task). Do not execute actions; classify only.'
config={'model_id':MODEL_ID,'seed':SEED,'epochs':3,'lora_r':8,'batch_size':1,'gradient_accumulation':8,'max_length':256,'learning_rate':0.0002,'source_branch':'feat/hermes-colab-lab-20261012','source_commit':'e9bbf751e1961a24243691d3b981c1d23f045b31','dataset_kind':'synthetic command classification','acceptance_accuracy':0.95,'production_accepted':False,'persistence_mode':'stage_then_existing_drive_connector','gpu':torch.cuda.get_device_name(0)}
config['packages']={p:version(p) for p in ['torch','transformers','peft','accelerate','datasets','bitsandbytes','safetensors']}
(run_dir/'config.json').write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding='utf-8')
print('Lab staging directory:',run_dir)


In [ ]:
# 3. Independent phrases, not random splitting of repeated examples
from datasets import Dataset
train_phrases={
'FIND':['ค้นหางานค้างใน Greenhouse','หางานที่เกี่ยวกับเว็บไซต์','ค้นเอกสารงานเมื่อวาน','หาข้อมูลของงานเดิม','ค้นบันทึกที่เกี่ยวกับร้านค้า','หางานตามเลข WORK-123','ค้นหาผลงานที่เก็บไว้','หาประวัติของงานนี้','ค้นรายการงานที่ค้าง','หาบันทึกการตัดสินใจ','ค้นไฟล์ของโครงการ','ค้นงานที่มีชื่อ HERMES'],
'INTAKE':['รับคำขอสร้างภาพ','รับโจทย์ใหม่จากเจ้าของ','บันทึกคำขอใหม่เข้าหน้ารับเรื่อง','รับบรีฟทำเว็บไซต์','รับรายละเอียดงานใหม่','ลงรับคำขอทำสไลด์','รับคำร้องใหม่ไว้ก่อน','รับเรื่องการออกแบบโปสเตอร์','รับความต้องการของลูกค้า','บันทึกบรีฟที่เพิ่งส่งมา','รับคำขอแก้เอกสารใหม่','รับเรื่องนี้เข้าสู่การรับงาน'],
'RETURN':['ส่งผลลัพธ์กลับ HALL','ส่งไฟล์ที่เสร็จกลับให้เจ้าของ','คืนรายงานไปยังผู้ขอ','ส่งชิ้นงานกลับต้นทาง','ส่งคำตอบกลับไปที่ฮอล','คืนผลทดสอบให้ผู้สั่ง','ส่งผลของงานกลับมา','ส่งหลักฐานกลับผู้รับ','นำผลงานกลับไปยังหน้ารับผล','ส่งไฟล์กลับให้บิ๊ก','คืนข้อมูลที่ทำเสร็จให้ต้นทาง','ส่งผลลัพธ์ที่ได้กลับฮอล'],
'STATUS':['ตรวจสถานะงาน','งานนี้ถึงขั้นไหนแล้ว','ดูความคืบหน้าของงานเดิม','บอกสถานะปัจจุบันของงาน','งานที่ส่งไปกำลังทำอยู่ไหม','ดูว่างานเสร็จหรือยัง','ตรวจว่างานนี้รออะไร','ขอดูสถานะ WORK-123','งานนี้ติดอยู่ตรงไหน','ดูขั้นตอนล่าสุดของงาน','เช็กความคืบหน้าของโครงการ','ตอนนี้งานอยู่สถานะอะไร'],
'VERIFY':['ตรวจผลทดสอบ','พิสูจน์ว่าไฟล์ใช้งานได้จริง','ตรวจความถูกต้องของผลลัพธ์','ตรวจหลักฐานที่ส่งกลับมา','ตรวจว่าสูตรคำนวณถูกไหม','ทดสอบคำตอบของโมเดล','ตรวจว่าผลงานตรงตามเกณฑ์','เช็กว่าไฟล์ที่บันทึกอ่านกลับได้','ตรวจคุณภาพของชิ้นงาน','ทดสอบว่าแอปทำงานได้จริง','ยืนยันผลจากหลักฐานจริง','ตรวจผลว่าผ่านเกณฑ์หรือไม่'],
'REPORT':['จัดทำรายงาน','สรุปผลการทำงานเป็นรายงาน','เขียนสรุปผลโครงการ','ทำรายงานจากข้อมูลที่มี','เรียบเรียงผลทดสอบเป็นสรุป','จัดทำบันทึกสรุปประจำวัน','เขียนรายงานการตรวจสอบ','สรุปข้อค้นพบลงเอกสาร','ทำสรุปสำหรับเจ้าของงาน','เขียนบทสรุปของงานนี้','จัดทำรายงานความก้าวหน้า','สรุปข้อมูลเป็นรายงานอ่านง่าย'],
'DISCOVER':['ค้นหาเครื่องมือที่รองรับ','หาเครื่องมือสำหรับแปลงไฟล์','ดูว่ามีเครื่องมือใดสร้างภาพได้','ค้นความสามารถที่ใช้ทำงานนี้ได้','หาเครื่องมือที่แก้เอกสารได้','ค้นตัวช่วยสำหรับงานวิดีโอ','ดูว่าตัวไหนรองรับการอ่านไฟล์','หาเครื่องมือที่เชื่อม Drive ได้','ค้นเครื่องมือสำหรับทำกราฟ','เลือกเครื่องมือที่รองรับงานนี้','หาเครื่องมือสำหรับทดสอบแอป','ค้นว่ามีเครื่องมือทำสไลด์ไหม'],
'EXECUTE':['เริ่มงานที่ได้รับมอบหมาย','ลงมือทำงานที่อนุมัติแล้ว','ดำเนินการตามแผนที่ตกลง','ทำงานตามบรีฟที่ยืนยันแล้ว','เริ่มประมวลผลงานที่มอบให้','ลงมือแก้ไฟล์ตามคำสั่ง','รันงานที่ผ่านการอนุมัติ','ทำขั้นตอนที่ได้รับมอบหมาย','ดำเนินงานนี้ตามขอบเขตเดิม','เริ่มสร้างชิ้นงานตามแผน','ลงมือทำงานที่ตกลงไว้','ดำเนินการงานที่เจ้าของสั่งไว้']}
validation_phrases={
'FIND':['ช่วยค้นแฟ้มงานเก่าของฉัน','ตามหาบันทึกของโครงการก่อนหน้า'],
'INTAKE':['ช่วยลงรับบรีฟใหม่ให้ที','มีโจทย์ใหม่ ขอฝากรับเรื่องไว้'],
'RETURN':['นำไฟล์ที่ทำแล้วส่งคืนผู้ขอ','ส่งคำตอบที่ได้กลับต้นทางด้วย'],
'STATUS':['ขอดูว่างานที่ฝากไว้ไปถึงไหน','งานเก่านี้กำลังรอขั้นตอนไหน'],
'VERIFY':['ตรวจสอบความแม่นยำของคำตอบนี้','พิสูจน์จากหลักฐานว่าการทดสอบผ่าน'],
'REPORT':['เรียบเรียงข้อมูลนี้เป็นรายงานสั้น','ทำเอกสารสรุปข้อค้นพบให้ที'],
'DISCOVER':['ต้องใช้ตัวช่วยอะไรเพื่อแปลงรูป','มีเครื่องมือไหนรองรับงานเสียงบ้าง'],
'EXECUTE':['เริ่มทำตามแผนที่ผมยืนยันได้เลย','จัดการงานที่มอบให้ตามข้อตกลง']}
test_phrases={
'FIND':['ตามหาไฟล์งานออกแบบครั้งก่อนให้หน่อย','อยากได้รายการงานเก่าที่มีคำว่าร้านค้า','ค้นประวัติการแก้ไขของโครงการนี้ให้ที','ช่วยค้นเอกสารที่เก็บในแฟ้มงานเดิม','หางานค้างทั้งหมดที่เคยบันทึก'],
'INTAKE':['มีบรีฟใหม่เข้ามา ช่วยรับเรื่องก่อน','ลงทะเบียนรับคำขอออกแบบชิ้นใหม่','ช่วยจดความต้องการใหม่ไว้ที่จุดรับงาน','รับรายละเอียดของงานที่เพิ่งส่งให้','เปิดรับโจทย์ใหม่นี้เข้าสู่การรับเรื่อง'],
'RETURN':['เอาผลที่ทำเสร็จส่งกลับฮอลหน่อย','ส่งมอบไฟล์ผลลัพธ์กลับผู้ว่าจ้าง','คืนหลักฐานการทำงานไปยังเจ้าของคำขอ','ส่งคำตอบชุดนี้กลับไปยังต้นทาง','นำผลสำเร็จกลับมาที่หน้ารับผล'],
'STATUS':['ตอนนี้งานที่ฝากไว้คืบหน้าเท่าไร','มีขั้นตอนไหนค้างอยู่ในงานนี้','งานที่สั่งไปเมื่อวานเสร็จถึงไหน','ช่วยดูว่างานเดิมกำลังทำหรือรออยู่','เช็กสถานะล่าสุดของงานให้ดูหน่อย'],
'VERIFY':['ช่วยทดสอบว่าผลลัพธ์นี้ใช้ได้จริง','เทียบคำตอบกับเกณฑ์ว่าถูกต้องไหม','ตรวจไฟล์หลังบันทึกว่าข้อมูลยังครบ','ช่วยพิสูจน์ว่าระบบทำงานสำเร็จจากหลักฐาน','ตรวจสอบความถูกต้องของสูตรในชิ้นงาน'],
'REPORT':['เขียนบทสรุปจากผลที่ได้ให้เจ้าของอ่าน','จัดข้อมูลทั้งหมดเป็นรายงานหนึ่งหน้า','รวบรวมข้อค้นพบแล้วเขียนรายงาน','ทำเอกสารรายงานผลการทดลองรอบนี้','สรุปผลตรวจสอบเป็นข้อความอ่านง่าย'],
'DISCOVER':['งานนี้มีตัวช่วยอะไรให้เลือกใช้บ้าง','ช่วยหาตัวที่สามารถตัดต่อวิดีโอได้','มีเครื่องมือใดเหมาะกับการแปลงเอกสาร','สำรวจความสามารถของเครื่องมือสำหรับงานนี้','หาเครื่องมือที่ทำตารางและกราฟได้'],
'EXECUTE':['ตามแผนที่อนุมัติไว้ เริ่มลงมือได้','ทำงานที่มอบหมายต่อจนได้ชิ้นงาน','ลงมือแก้ปัญหานี้ภายใต้ขอบเขตที่ตกลง','รันขั้นตอนที่ได้รับอนุญาตแล้วได้เลย','เริ่มผลิตชิ้นงานตามคำสั่งเดิม']}
train_rows=[{'query':prefix+q,'label':label} for label,qs in train_phrases.items() for q in qs for prefix in ['', 'โก ']]
validation_rows=[{'query':q,'label':label} for label,qs in validation_phrases.items() for q in qs]
test_rows=[{'query':q,'label':label} for label,qs in test_phrases.items() for q in qs]
sets=[{r['query'].strip() for r in rows} for rows in [train_rows,validation_rows,test_rows]]
assert not (sets[0]&sets[1] or sets[0]&sets[2] or sets[1]&sets[2]), 'Data leakage'
assert all(len(s)==len(rows) for s,rows in zip(sets,[train_rows,validation_rows,test_rows]))
for name,rows in [('train',train_rows),('validation',validation_rows),('test',test_rows)]:
    p=run_dir/(name+'.jsonl')
    p.write_text(''.join(json.dumps(r,ensure_ascii=False)+'\n' for r in rows),encoding='utf-8')
print('Split counts:',len(train_rows),len(validation_rows),len(test_rows),'; no exact overlap')


In [ ]:
# 4. Load frozen base revision and prepare real 4-bit QLoRA
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig, BitsAndBytesConfig, TrainingArguments, Trainer, set_seed
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
set_seed(SEED)
base_config=AutoConfig.from_pretrained(MODEL_ID,trust_remote_code=False)
from huggingface_hub import HfApi
base_revision=HfApi().model_info(MODEL_ID).sha
assert base_revision, 'Could not resolve immutable base revision'
config['base_revision']=base_revision
(run_dir/'config.json').write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding='utf-8')
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=base_revision,trust_remote_code=False)
if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
tokenizer.padding_side='right'
bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16)
def load_base():
    return AutoModelForCausalLM.from_pretrained(MODEL_ID,revision=base_revision,quantization_config=bnb,device_map={'':0},torch_dtype=torch.float16,trust_remote_code=False)
model=load_base()
model=prepare_model_for_kbit_training(model,use_gradient_checkpointing=True)
model=get_peft_model(model,LoraConfig(r=8,lora_alpha=16,lora_dropout=0.05,target_modules=['q_proj','k_proj','v_proj','o_proj'],bias='none',task_type='CAUSAL_LM'))
model.config.use_cache=False
model.print_trainable_parameters()
def prompt_messages(query): return [{'role':'system','content':SYSTEM},{'role':'user','content':query}]
def encode_row(row):
    prompt=tokenizer.apply_chat_template(prompt_messages(row['query']),tokenize=False,add_generation_prompt=True)
    complete=tokenizer.apply_chat_template(prompt_messages(row['query'])+[{'role':'assistant','content':row['label']}],tokenize=False)
    prefix=tokenizer(prompt,add_special_tokens=False)['input_ids']
    ids=tokenizer(complete,add_special_tokens=False)['input_ids']
    assert ids[:len(prefix)]==prefix and len(ids)<=256
    labels=[-100]*len(prefix)+ids[len(prefix):]
    assert any(x!=-100 for x in labels)
    return {'input_ids':ids,'attention_mask':[1]*len(ids),'labels':labels}
train_dataset=Dataset.from_list([encode_row(r) for r in train_rows])
validation_dataset=Dataset.from_list([encode_row(r) for r in validation_rows])
def collate(rows):
    width=max(len(r['input_ids']) for r in rows)
    return {key:torch.tensor([r[key]+[pad]*(width-len(r[key])) for r in rows],dtype=torch.long) for key,pad in [('input_ids',tokenizer.pad_token_id),('attention_mask',0),('labels',-100)]}
print('MODEL AND DATA READY',base_revision)


In [ ]:
# 5. Baseline and training. Checkpoints are staged locally and included in the Drive export.
import traceback
@torch.inference_mode()
def predict(query):
    inputs=tokenizer(tokenizer.apply_chat_template(prompt_messages(query),tokenize=False,add_generation_prompt=True),return_tensors='pt').to('cuda')
    generated=model.generate(**inputs,max_new_tokens=12,do_sample=False,pad_token_id=tokenizer.pad_token_id,eos_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(generated[0,inputs['input_ids'].shape[1]:],skip_special_tokens=True).strip()
def evaluate_rows(rows):
    predictions=[]
    model.eval()
    for row in rows:
        start=time.monotonic()
        output=predict(row['query'])
        predictions.append({**row,'prediction':output,'correct':output==row['label'],'seconds':round(time.monotonic()-start,3)})
    per_class={label:{'correct':sum(r['correct'] for r in predictions if r['label']==label),'total':sum(r['label']==label for r in predictions)} for label in train_phrases}
    return {'correct':sum(r['correct'] for r in predictions),'total':len(predictions),'accuracy':sum(r['correct'] for r in predictions)/len(predictions),'per_class':per_class},predictions
with model.disable_adapter():
    baseline,base_predictions=evaluate_rows(test_rows)
(run_dir/'baseline.json').write_text(json.dumps(baseline,ensure_ascii=False,indent=2),encoding='utf-8')
(run_dir/'baseline_predictions.jsonl').write_text(''.join(json.dumps(r,ensure_ascii=False)+'\n' for r in base_predictions),encoding='utf-8')
print('Baseline:',baseline['correct'],'/',baseline['total'])
model.train()
args=TrainingArguments(output_dir=str(run_dir/'checkpoints'),per_device_train_batch_size=1,per_device_eval_batch_size=1,gradient_accumulation_steps=8,num_train_epochs=3,learning_rate=2e-4,logging_steps=4,save_strategy='steps',save_steps=24,save_total_limit=3,eval_strategy='epoch',report_to='none',fp16=True,gradient_checkpointing=True,optim='adamw_torch',seed=SEED,data_seed=SEED,remove_unused_columns=False)
trainer=Trainer(model=model,args=args,train_dataset=train_dataset,eval_dataset=validation_dataset,data_collator=collate,processing_class=tokenizer)
try:
    started=time.monotonic()
    train_result=trainer.train()
    train_metrics={**train_result.metrics,'elapsed_seconds':round(time.monotonic()-started,2),'global_step':trainer.state.global_step}
    adapter_dir=run_dir/'adapter'
    trainer.model.save_pretrained(str(adapter_dir),safe_serialization=True)
    tokenizer.save_pretrained(str(adapter_dir))
    (run_dir/'training_metrics.json').write_text(json.dumps(train_metrics,indent=2),encoding='utf-8')
    (run_dir/'training_log.json').write_text(json.dumps(trainer.state.log_history,indent=2),encoding='utf-8')
    print('TRAINING FINISHED:',train_metrics)
except Exception:
    (run_dir/'training_failure.txt').write_text(traceback.format_exc(),encoding='utf-8')
    raise


In [ ]:
# 6. Reload saved local adapter, test independently, then archive for Drive
import gc
from safetensors import safe_open
assert (adapter_dir/'adapter_model.safetensors').stat().st_size>0
with safe_open(str(adapter_dir/'adapter_model.safetensors'),framework='pt',device='cpu') as f:
    tensor_keys=list(f.keys())
    assert tensor_keys and all(torch.isfinite(f.get_tensor(k)).all() for k in tensor_keys)
saved_adapter_config=json.loads((adapter_dir/'adapter_config.json').read_text())
assert saved_adapter_config['r']==8
# Destroy trained in-memory model: prediction must use the saved files.
del trainer,model
gc.collect()
torch.cuda.empty_cache()
model=PeftModel.from_pretrained(load_base(),str(adapter_dir),is_trainable=False)
model.config.use_cache=True
metrics,predictions=evaluate_rows(test_rows)
metrics.update({'threshold':0.95,'lab_acceptance':'PASS' if metrics['accuracy']>=0.95 else 'FAIL','synthetic_test':True,'production_accepted':False,'reloaded_saved_adapter':True,'reloaded_from_drive':False,'base_accuracy':baseline['accuracy'],'accuracy_delta':metrics['accuracy']-baseline['accuracy'],'peak_gpu_gib':round(torch.cuda.max_memory_allocated()/2**30,3)})
(run_dir/'evaluation.json').write_text(json.dumps(metrics,ensure_ascii=False,indent=2),encoding='utf-8')
(run_dir/'predictions.jsonl').write_text(''.join(json.dumps(r,ensure_ascii=False)+'\n' for r in predictions),encoding='utf-8')
manifest=[]
for path in sorted(run_dir.rglob('*')):
    if path.is_file() and path.name not in ['manifest.json','readback.json']:
        digest=hashlib.sha256()
        with path.open('rb') as f:
            for block in iter(lambda:f.read(1024*1024),b''): digest.update(block)
        manifest.append({'path':str(path.relative_to(run_dir)),'bytes':path.stat().st_size,'sha256':digest.hexdigest()})
(run_dir/'manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
for item in json.loads((run_dir/'manifest.json').read_text()):
    path=run_dir/item['path']; digest=hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): digest.update(block)
    assert path.stat().st_size==item['bytes'] and digest.hexdigest()==item['sha256']
assert len([json.loads(line) for line in (run_dir/'predictions.jsonl').read_text().splitlines()])==len(test_rows)
checkpoint_dirs=list((run_dir/'checkpoints').glob('checkpoint-*'))
assert checkpoint_dirs, 'Checkpoint missing'
readback={'files_verified':len(manifest),'checkpoints':len(checkpoint_dirs),'adapter_tensor_count':len(tensor_keys),'evaluation':json.loads((run_dir/'evaluation.json').read_text()),'status':'PASS'}
(run_dir/'readback.json').write_text(json.dumps(readback,ensure_ascii=False,indent=2),encoding='utf-8')
print('SAVED ADAPTER RELOADED; LOCAL FILE READBACK PASSED')
print(json.dumps(metrics,ensure_ascii=False,indent=2))
print('Run folder:',run_dir)

import shutil
archive=shutil.make_archive(str(run_dir)+"-artifacts","zip",root_dir=str(run_dir))
print("ARTIFACT_ARCHIVE:",archive)
print("ARCHIVE_SHA256:",hashlib.sha256(Path(archive).read_bytes()).hexdigest())


In [ ]:
# Export only this Lab's generated artifacts
from google.colab import files
files.download(archive)
print('Artifact archive handed to browser download:', Path(archive).name)


## LiteRT-LM status
The trained output above is a LoRA adapter, not a `.litertlm` model. The separate merge/export notebook must load the saved adapter, merge it with the exact base revision, and use Google's supported Qwen2.5 conversion recipe. Qwen2.5 has an official LiteRT-LM model family, but this fine-tuned adapter has not been converted or tested on the owner's Xiaomi. The conversion environment should be isolated from the training environment. Never rename the adapter or merged checkpoint to `.litertlm` and claim conversion.
